# Top-View LPG Cylinder Classifier — v1

## Overview
Trains the top-view branch of the three-view ensemble classifier: given a crop of the
cylinder photographed from directly above, predict the brand (bharat / hp / indane). This
notebook builds and trains an EfficientNetB2 + SpatialAttention model on a top-view-only
crop dataset, mirroring the architecture used for the main (side-view) classifier but with
augmentation tuned for the flatter, less perspective-distorted geometry of a top-down shot.
It is one of three classifier notebooks (side, bottom-ring, top-view) whose outputs are
combined by `src/predict_ensemble.py`.

## How to Run
1. **Runtime:** Colab GPU (T4 is sufficient — EfficientNetB2 at 224x224, batch size 32,
   40 epochs). CPU-only will be impractically slow for the training loop.
2. **Upload/mount:** Mount Google Drive (cell 1) for optional dataset/checkpoint access.
   In cell "Upload / unzip dataset", either upload a zip directly (`UPLOAD_LOCAL = True`,
   the default) or set `UPLOAD_LOCAL = False` and point `DRIVE_ZIP_PATH` at a zip already
   in Drive. The zip must extract to `topview/train/{bharat,hp,indane}` and
   `topview/valid/{bharat,hp,indane}`.
3. **Execution order:** Run cells top to bottom. No manual reordering needed once the
   dataset zip is available; the only user-provided input is the zip itself (upload prompt
   or `DRIVE_ZIP_PATH`).
4. **Expected outputs:** `classifier_topview_v1.pth` checkpoint, `training_history_topview_v1.json`,
   a training-curve plot, and a confusion-matrix plot — see the Output Files section near the
   end of the notebook.

## Model / Dataset Info
| | |
|---|---|
| Architecture | EfficientNetB2 backbone + SpatialAttention head (`LPGClassifierAttention`, same shape as `src/predict.py`'s side-view model, `num_classes=3`) |
| Dataset | Top-view crops only, 3-class (`bharat`, `hp`, `indane`) |
| Classes | bharat, hp, indane — no `unknown` in this dataset |
| Expected accuracy | Reported via `best_val_acc` at training time; per project records the shipped `classifier_topview_v1.pth` checkpoint achieved **77.78%** val accuracy |

## Current Status
This notebook's code builds a **B2 + spatial-attention** model, but the currently shipped
checkpoint (`models/classifier_topview_v1.pth`) is actually a **frozen-backbone EfficientNetB0**
— the architecture in this notebook does not match what's deployed. The loader in
`src/predict_ensemble.py` is correct for the shipped B0 checkpoint; this notebook is left
as-is (not "fixed") since the constraint is documentation-only and the discrepancy is a known,
tracked issue (see project `CLAUDE.md` — highest-priority item is reconciling this). Training
loop, evaluation, and Drive-save logic otherwise mirror the other classifier notebooks and are
functionally complete.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Upload / unzip dataset

Expects a zip that extracts to `topview/train/{bharat,hp,indane}` and `topview/valid/{bharat,hp,indane}`.

Either upload the zip directly, or point `DRIVE_ZIP_PATH` at a zip already sitting in Drive.

In [ ]:
import os

os.makedirs('/content/dataset', exist_ok=True)

# Option A — upload a zip from local disk
UPLOAD_LOCAL = True

# Option B — copy a zip already in Drive (set UPLOAD_LOCAL = False to use this)
DRIVE_ZIP_PATH = '/content/drive/MyDrive/LPG Cylinder Detection and Classification/Dataset/topview.zip'  # ← UPDATE THIS PATH — dataset zip location in Drive

if UPLOAD_LOCAL:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
else:
    import shutil
    zip_name = 'topview.zip'
    shutil.copy(DRIVE_ZIP_PATH, f'/content/{zip_name}')

print(f'Using zip: {zip_name}')

In [ ]:
import zipfile

with zipfile.ZipFile(f'/content/{zip_name}', 'r') as zf:
    zf.extractall('/content/dataset/topview')

TRAIN_DIR = '/content/dataset/topview/train'
VALID_DIR = '/content/dataset/topview/valid'
CLASSES   = ['bharat', 'hp', 'indane']

for split_dir in (TRAIN_DIR, VALID_DIR):
    print(split_dir)
    for c in CLASSES:
        p = os.path.join(split_dir, c)
        n = len(os.listdir(p)) if os.path.exists(p) else 0
        print(f'  {c}: {n}')

## 3. Imports

In [ ]:
import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms as T

from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 4. Transforms

Same augmentation recipe as v7, except `RandomPerspective` distortion scale is reduced from `0.5` to `0.2` — top-view crops don't exhibit the strong perspective skew seen in angled kiosk shots, so heavy perspective jitter would push the augmented distribution away from real top-view data.

In [ ]:
IMG_SIZE = 224
PERSPECTIVE_DISTORTION = 0.2  # v7 used 0.5; reduced for top-view's flatter geometry

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.RandomPerspective(distortion_scale=PERSPECTIVE_DISTORTION, p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

valid_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## 5. Dataset + weighted sampler

`WeightedRandomSampler` compensates for class imbalance across bharat/hp/indane (mirrors the sampling strategy used for the 4-class training runs).

In [ ]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
valid_dataset = datasets.ImageFolder(VALID_DIR, transform=valid_transform)

assert train_dataset.classes == CLASSES, f'Class order mismatch: {train_dataset.classes} vs {CLASSES}'
assert valid_dataset.classes == CLASSES, f'Class order mismatch: {valid_dataset.classes} vs {CLASSES}'

class_counts = np.bincount(train_dataset.targets, minlength=len(CLASSES))
print('Train class counts:', dict(zip(CLASSES, class_counts)))

class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[train_dataset.targets]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_dataset)} images | Valid: {len(valid_dataset)} images')

## 6. Model — EfficientNetB2 + SpatialAttention

Same architecture as `classifier_best_v7_3class.pth` / `src/predict.py`'s `LPGClassifierAttention`, with `num_classes=3`.

In [ ]:
class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv    = nn.Conv2d(1, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_pool = x.mean(dim=1, keepdim=True)
        max_pool = x.max(dim=1, keepdim=True).values
        pooled   = avg_pool + max_pool
        att_map  = self.sigmoid(self.conv(pooled))
        return x * att_map


class LPGClassifierAttention(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        backbone        = models.efficientnet_b2(weights='IMAGENET1K_V1')
        self.features   = backbone.features
        self.avgpool    = backbone.avgpool
        feat_dim        = backbone.classifier[1].in_features
        self.attention  = SpatialAttention()
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        feat_maps = self.features(x)
        attended  = self.attention(feat_maps)
        pooled    = self.avgpool(attended)
        flat      = torch.flatten(pooled, 1)
        return self.classifier(flat)


model = LPGClassifierAttention(num_classes=len(CLASSES)).to(device)
print(model.__class__.__name__, '- params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7. Training setup

In [ ]:
EPOCHS = 40
LR = 3e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 8. Training loop

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

    return total_loss / total, 100.0 * correct / total

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_state = None

start = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(valid_loader, train=False)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

    marker = ' *' if is_best else ''
    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'train_loss {train_loss:.4f} train_acc {train_acc:.2f}% | '
          f'val_loss {val_loss:.4f} val_acc {val_acc:.2f}%{marker}')

elapsed = time.time() - start
print(f'\nTraining done in {elapsed/60:.1f} min | best val acc: {best_val_acc:.2f}%')

model.load_state_dict(best_state)

## 9. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['val_acc'], label='val')
axes[1].set_title('Accuracy (%)')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Evaluation + confusion matrix

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Top-View Classifier v1 — val acc {best_val_acc:.2f}%')
plt.tight_layout()
plt.show()

## 11. Save checkpoint + push to Drive

In [ ]:
import json

CHECKPOINT_NAME = 'classifier_topview_v1.pth'
LOCAL_CKPT_PATH = f'/content/{CHECKPOINT_NAME}'

checkpoint = {
    'model_state_dict':     model.state_dict(),
    'best_val_acc':         best_val_acc,
    'classes':              CLASSES,
    'architecture':         'efficientnet_b2_attention',
    'confidence_threshold': 0.60,
    'epochs':               EPOCHS,
    'batch_size':            BATCH_SIZE,
    'lr':                    LR,
    'perspective_distortion': PERSPECTIVE_DISTORTION,
    'img_size':              IMG_SIZE,
    'notes':                 'Top-view-only dataset; reduced RandomPerspective vs v7 (0.2 vs 0.5)',
}

torch.save(checkpoint, LOCAL_CKPT_PATH)
print(f'Saved checkpoint locally: {LOCAL_CKPT_PATH}')

with open('/content/training_history_topview_v1.json', 'w') as f:
    json.dump(history, f, indent=2)

In [ ]:
import shutil

DRIVE_SAVE_DIR = '/content/drive/MyDrive/LPG Cylinder Detection and Classification/Classifier/topview_v1'  # ← UPDATE THIS PATH
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

shutil.copy(LOCAL_CKPT_PATH, os.path.join(DRIVE_SAVE_DIR, CHECKPOINT_NAME))
shutil.copy('/content/training_history_topview_v1.json', os.path.join(DRIVE_SAVE_DIR, 'training_history_topview_v1.json'))

print(f'Saved to Drive: {DRIVE_SAVE_DIR}')
print(f'  - {CHECKPOINT_NAME}')
print(f'  - training_history_topview_v1.json')
print(f'\nBest val acc: {best_val_acc:.2f}%')

## Output Files

| File | Saved to | Contents / purpose |
|---|---|---|
| `classifier_topview_v1.pth` | `/content/` then Drive `Classifier/topview_v1/` | Model checkpoint dict — `model_state_dict`, `best_val_acc`, `classes`, `architecture`, `confidence_threshold`, plus training metadata (`epochs`, `batch_size`, `lr`, `perspective_distortion`, `img_size`, `notes`). Consumed by `src/predict_ensemble.py`'s top-view branch. |
| `training_history_topview_v1.json` | `/content/` then Drive `Classifier/topview_v1/` | Per-epoch `train_loss`/`train_acc`/`val_loss`/`val_acc` history, for later re-plotting or comparison across runs. |
| Training curves plot | Displayed inline only (not saved to disk) | Loss/accuracy vs. epoch, train vs. val. |
| Confusion matrix plot | Displayed inline only (not saved to disk) | Per-class prediction breakdown on the validation set at `best_val_acc`. |